In [73]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
from itertools import product
import plotly.express as px
import warnings
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn import metrics
from sklearn.metrics import v_measure_score
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import time
from sklearn.decomposition import PCA, KernelPCA
import umap.umap_ as umap

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [74]:
# Загрузка датасета
df = pd.read_excel('predictive_maintenance.xlsx')
# Подготовка данных (удаление 'Failure Type', фильтрация 'TargetF'>0, сортировка)
df = df.drop('Failure Type', axis=1).loc[df['TargetF'] > 0].sort_values('TargetF')

columns = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
X = df[columns]
y = df['TargetF']
y = y - 1


In [75]:
# StandardScaler()
scaler = preprocessing.StandardScaler()
scaled_features = scaler.fit_transform(X)
X_scaled = pd.DataFrame(scaled_features, index=X.index, columns=X.columns)

In [76]:
# --- Вспомогательные функции для Grid Search ---

def find_optimal_k_silhouette(X_data, max_k=12):
    """Определяет оптимальное k на основе Silhouette Score для алгоритмов, требующих k."""
    silhouette_scores = []
    k_range = range(2, min(max_k + 1, len(X_data) - 1))

    if len(X_data) < 2 or not k_range:
        return 1

    for k in k_range:
        # Для KMeans
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=123)
        clusters = kmeans.fit_predict(X_data)
        if len(np.unique(clusters)) > 1:
            silhouette_scores.append(metrics.silhouette_score(X_data, clusters))
        else:
            silhouette_scores.append(-1)

    if not silhouette_scores or all(s == -1 for s in silhouette_scores):
        return 1

    optimal_k_index = np.argmax(silhouette_scores)
    optimal_k = k_range[optimal_k_index]
    return optimal_k

In [77]:
# Оценка кластеризации:
def evaluate_clustering(y_true, clusters):
    if len(np.unique(clusters)) <= 1 or len(np.unique(y_true)) <= 1:
        return 0.0, 0.0, 0.0, 0.0
    h = metrics.homogeneity_score(y_true, clusters)
    c = metrics.completeness_score(y_true, clusters)
    r = metrics.rand_score(y_true, clusters)
    v = v_measure_score(y_true, clusters)
    return h, c, r, v

In [78]:
def find_best_dbscan_params(X_data, y_true, eps_range, min_samples_range, max_k_target=12):
    best_score = -1
    best_params = None
    best_clusters = None
    best_k = 0

    for eps in eps_range:
        for min_samples in min_samples_range:
            dbscan = DBSCAN(eps=eps, min_samples=min_samples)
            clusters = dbscan.fit_predict(X_data)

            n_clusters_dbscan = len(set(clusters)) - (1 if -1 in clusters else 0)  # Исключаем шум (-1)

            if n_clusters_dbscan > 1 and n_clusters_dbscan <= max_k_target and len(np.unique(clusters)) > 1:
                score = metrics.silhouette_score(X_data, clusters)
                if score > best_score:
                    best_score = score
                    best_params = {'eps': eps, 'min_samples': min_samples}
                    best_clusters = clusters
                    best_k = n_clusters_dbscan  # Фактическое число кластеров

    # Если не найдено кластеров >1, возвращаем параметры
    if best_params is None:
        dbscan = DBSCAN(eps=eps_range[0], min_samples=min_samples_range[0])
        clusters = dbscan.fit_predict(X_data)
        n_clusters_dbscan = len(set(clusters)) - (1 if -1 in clusters else 0)
        best_params = {'eps': eps_range[0], 'min_samples': min_samples_range[0]}
        best_clusters = clusters
        best_k = n_clusters_dbscan

    return best_params, best_clusters, best_k

In [79]:
# --- Главная функция Grid Search ---
all_results = []
max_k_for_silhouette = min(12, len(X_scaled) - 1)

# Диапазоны гиперпараметров для методов снижения размерности
pca_n_components_range = range(1, min(X_scaled.shape[1], 5) + 1)
kpca_n_components_range = range(1, min(X_scaled.shape[1], 5) + 1)
umap_n_components_range = range(1, min(X_scaled.shape[1], 5) + 1)
umap_n_neighbors_range = range(1, 6)  # От 1 до 5 соседей

# Диапазоны гиперпараметров для алгоритмов кластеризации
n_clusters_range = range(1, 13)  # Для KMeans, GMM, Agglomerative
dbscan_eps_range = np.arange(0.1, 1.1, 0.2)
dbscan_min_samples_range = range(2, 7)
agglomerative_linkages = ['ward', 'complete', 'average', 'single']

# Итерация по методам снижения размерности
dr_methods = {
    "Original Scaled Data": (None, X_scaled, 'N/A', X_scaled.shape[1]),
    "PCA": (PCA, {'n_components': pca_n_components_range}, 'n_components', 0),
    "KernelPCA": (KernelPCA, {
        'n_components': kpca_n_components_range,
        'kernel': ['sigmoid', 'linear', 'poly', 'rbf'],
        'gamma': [0.1]
    }, 'n_components, kernel', 0),
    "UMAP": (umap.UMAP, {
        'n_components': umap_n_components_range,
        'n_neighbors': umap_n_neighbors_range
    }, 'n_components, n_neighbors', 0)
}

# Для каждого набора данных (оригинальные или редуцированные)
for dr_name, (dr_class, dr_params_grid, dr_param_names_str, fixed_n_comp) in dr_methods.items():

    dr_param_combinations = []
    if dr_class is None:
        dr_param_combinations.append(({'transformer': None, 'params_str': 'N/A', 'n_components': fixed_n_comp}))
    else:
        # Генерируем все комбинации параметров для DR
        from itertools import product
        param_keys = dr_params_grid.keys()
        for values in product(*dr_params_grid.values()):
            dr_params = dict(zip(param_keys, values))

            # Условие для UMAP: n_neighbors >= n_components
            if dr_name == 'UMAP' and dr_params.get('n_neighbors', 0) < dr_params['n_components']:
                 if dr_params['n_components'] > 0:
                     dr_params['n_neighbors'] = dr_params['n_components']

            dr_param_combinations.append({
                'transformer': dr_class(**dr_params, random_state=123, verbose=False) if dr_name == 'UMAP' else dr_class(**dr_params, random_state=123),
                'params_str': ', '.join([f"{k}={v}" for k, v in dr_params.items()]),
                'n_components': dr_params.get('n_components', X_scaled.shape[1]),
                'n_neighbors': dr_params.get('n_neighbors', 'N/A')  # Для UMAP
            })

    for dr_config in dr_param_combinations:
        dr_transformer = dr_config['transformer']
        dr_params_str = dr_config['params_str']
        dr_n_components = dr_config['n_components']
        dr_n_neighbors = dr_config.get('n_neighbors', 'N/A')

        print(f"\n--- Применение DR: {dr_name} ({dr_params_str}) ---")

        X_current_data = X_scaled
        if dr_transformer is None:
            X_transformed = X_current_data
        else:
            try:
                X_transformed = dr_transformer.fit_transform(X_current_data)
            except Exception as e:
                print(f"Ошибка при DR {dr_name} ({dr_params_str}): {e}. Пропускаем.")
                continue

        X_transformed_df = pd.DataFrame(X_transformed, columns=[f'Comp_{i+1}' for i in range(X_transformed.shape[1])], index=X_scaled.index)

        # --- Перебор алгоритмов кластеризации ---
        clustering_algos = {
            "KMeans": (KMeans, {'n_clusters': n_clusters_range}),
            "GaussianMixture": (GaussianMixture, {'n_components': n_clusters_range}),
            "DBSCAN": (DBSCAN, {'eps': dbscan_eps_range, 'min_samples': dbscan_min_samples_range}),
            "AgglomerativeClustering": (AgglomerativeClustering, {'n_clusters': n_clusters_range, 'linkage': agglomerative_linkages})
        }

        for algo_name, (algo_class, algo_params_grid) in clustering_algos.items():
            algo_param_combinations = []
            param_keys = algo_params_grid.keys()
            for values in product(*algo_params_grid.values()):
                algo_param_combinations.append(dict(zip(param_keys, values)))

            for algo_params in algo_param_combinations:
                current_algo_params_str = ', '.join([f"{k}={v}" for k, v in algo_params.items()])

                clusters = None
                n_clusters_algo = 0
                optimal_k_found = 0

                try:
                    if algo_name == "KMeans":
                        optimal_k_found = find_optimal_k_silhouette(X_transformed, max_k=max_k_for_silhouette)
                        kmeans = KMeans(n_clusters=optimal_k_found, n_init=10, random_state=123)
                        clusters = kmeans.fit_predict(X_transformed)
                        n_clusters_algo = optimal_k_found
                        current_algo_params_str = f"n_clusters={optimal_k_found}"

                    elif algo_name == "GaussianMixture":
                        optimal_k_found = find_optimal_k_silhouette(X_transformed, max_k=max_k_for_silhouette)
                        gmm = GaussianMixture(n_components=optimal_k_found, random_state=123, n_init=10)
                        gmm.fit(X_transformed)
                        clusters = gmm.predict(X_transformed)
                        n_clusters_algo = optimal_k_found
                        current_algo_params_str = f"n_components={optimal_k_found}"

                    elif algo_name == "DBSCAN":
                        best_dbscan_params, clusters, n_clusters_algo = find_best_dbscan_params(X_transformed, y, dbscan_eps_range, dbscan_min_samples_range, max_k_target=max_k_for_silhouette)
                        current_algo_params_str = f"eps={best_dbscan_params['eps']:.1f}, min_samples={best_dbscan_params['min_samples']}"
                        optimal_k_found = n_clusters_algo  # Для DBSCAN это фактически найденное число кластеров

                    elif algo_name == "AgglomerativeClustering":
                        # Для Agglomerative будем искать n_clusters для каждого linkage
                        current_linkage = algo_params['linkage']

                        best_aggl_score = -1
                        best_aggl_k = 1
                        best_aggl_clusters = None

                        aggl_k_range = range(2, min(max_k_for_silhouette + 1, len(X_transformed) - 1))
                        if len(aggl_k_range) > 0:
                            for k in aggl_k_range:
                                aggl = AgglomerativeClustering(n_clusters=k, linkage=current_linkage)
                                aggl_clusters = aggl.fit_predict(X_transformed)
                                if len(np.unique(aggl_clusters)) > 1:
                                    score = metrics.silhouette_score(X_transformed, aggl_clusters)
                                    if score > best_aggl_score:
                                        best_aggl_score = score
                                        best_aggl_k = k
                                        best_aggl_clusters = aggl_clusters

                            # Если лучший k не найден (>1 кластера), используем 1
                            if best_aggl_clusters is None:
                                aggl = AgglomerativeClustering(n_clusters=1, linkage=current_linkage)
                                best_aggl_clusters = aggl.fit_predict(X_transformed)
                                best_aggl_k = 1
                        else: # Если данных недостаточно для k=2
                            aggl = AgglomerativeClustering(n_clusters=1, linkage=current_linkage)
                            best_aggl_clusters = aggl.fit_predict(X_transformed)
                            best_aggl_k = 1

                        clusters = best_aggl_clusters
                        n_clusters_algo = best_aggl_k
                        optimal_k_found = best_aggl_k
                        current_algo_params_str = f"n_clusters={best_aggl_k}, linkage={current_linkage}"

                    # Вычисление метрик качества
                    h, c, r, v = evaluate_clustering(y, clusters)

                    all_results.append({
                        'DR_Method': dr_name,
                        'DR_Hyperparameter': dr_params_str,
                        'DR_n_components': dr_n_components,
                        'DR_n_neighbors': dr_n_neighbors,
                        'Clustering_Algo': algo_name,
                        'Clustering_Hyperparameter': current_algo_params_str,
                        'Optimal_k_found': n_clusters_algo,
                        'Homogeneity': h,
                        'Completeness': c,
                        'Rand Score': r,
                        'V_measure':v
                    })
                    print(f"  {algo_name} ({current_algo_params_str}): H={h:.3f}, C={c:.3f}, R={r:.3f}, Actual k={n_clusters_algo}")

                except Exception as e:
                    print(f"  Ошибка при кластеризации {algo_name} ({current_algo_params_str}) на {dr_name} ({dr_params_str}): {e}. Пропускаем.")
                    pass


--- Применение DR: Original Scaled Data (N/A) ---
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  KMeans (n_clusters=2): H=0.079, C=0.341, R=0.363, Actual k=2
  GaussianMixture (n_components=2): H=0.086, C=0.311, R=0.406, Actual k=2
  GaussianMixture (n_components=2): H=0.086, C=0.311, R=0.406, Actual k=2
  GaussianMixture (n_components=2): H=0.086, 

In [80]:
# --- Сводная таблица результатов ---
results_df_summary = pd.DataFrame(all_results)
results_df_summary = results_df_summary.sort_values(by='V_measure', ascending=False, ignore_index=True)

print("\n--- Сводная таблица результатов Grid Search (Топ 20) ---")
print(results_df_summary.head(20).to_string())


--- Сводная таблица результатов Grid Search (Топ 20) ---
    DR_Method                          DR_Hyperparameter  DR_n_components DR_n_neighbors  Clustering_Algo Clustering_Hyperparameter  Optimal_k_found  Homogeneity  Completeness  Rand Score  V_measure
0   KernelPCA  n_components=4, kernel=sigmoid, gamma=0.1                4            N/A  GaussianMixture            n_components=3                3     0.489130      0.674191    0.793004   0.566940
1   KernelPCA  n_components=4, kernel=sigmoid, gamma=0.1                4            N/A  GaussianMixture            n_components=3                3     0.489130      0.674191    0.793004   0.566940
2   KernelPCA  n_components=4, kernel=sigmoid, gamma=0.1                4            N/A  GaussianMixture            n_components=3                3     0.489130      0.674191    0.793004   0.566940
3   KernelPCA  n_components=4, kernel=sigmoid, gamma=0.1                4            N/A  GaussianMixture            n_components=3               

In [81]:
# --- Выбор лучшей комбинации ---
best_result = results_df_summary.loc[0]
print("\n--- Лучшая комбинация метода снижения размерности и алгоритма кластеризации ---")
print(best_result)


--- Лучшая комбинация метода снижения размерности и алгоритма кластеризации ---
DR_Method                                                    KernelPCA
DR_Hyperparameter            n_components=4, kernel=sigmoid, gamma=0.1
DR_n_components                                                      4
DR_n_neighbors                                                     N/A
Clustering_Algo                                        GaussianMixture
Clustering_Hyperparameter                               n_components=3
Optimal_k_found                                                      3
Homogeneity                                                    0.48913
Completeness                                                  0.674191
Rand Score                                                    0.793004
V_measure                                                      0.56694
Name: 0, dtype: object


Из результата видно, что наилучший результат дает применение KernelPCA с kernel=sigmoid + алгоритм GaussianMixture

Попробую улучшить результат: перебрать больше гиперпараметров для KernelPCA с kernel=sigmoid + GaussianMixture

In [82]:
# Вспомогательные функции для визуализации
def plot_3d(X_df, y_true, clusters, algo_name, params_str, feature_names):
    # Создаем DataFrame для Plotly
    X_df['clusters'] = clusters
    X_df['true_classes'] = y_true

    # Определяем фиксированные цвета для кластеров
    unique_clusters = X_df['clusters'].unique()
    colors = ['red', 'green', 'blue', 'yellow', 'purple']

    # Создаем 3D график
    fig = px.scatter_3d(X_df, x=feature_names[0], y=feature_names[1], z=feature_names[2],
                        color='clusters',
                        hover_data=['true_classes'],
                        title=f"{algo_name} ({params_str})",
                        opacity=0.7,
                        color_discrete_sequence=[colors[i % len(colors)] for i in range(len(unique_clusters))])

    fig.update_traces(marker=dict(size=5))
    fig.show()

In [83]:
# Поиск оптимального k для методов, требующих k (используем диапазон и считаем метрику целевой V_measure)
def find_best_k_by_metric(X_data, y_true, algo_type, k_range, extra_params=None):
    best_k = None
    best_score = -np.inf
    best_labels = None
    for k in k_range:
        try:
            if algo_type == 'KMeans':
                model = KMeans(n_clusters=k, n_init=10, random_state=123)
                labels = model.fit_predict(X_data)
            elif algo_type == 'GaussianMixture':
                model = GaussianMixture(n_components=k, random_state=123, n_init=10)
                model.fit(X_data)
                labels = model.predict(X_data)
            elif algo_type == 'Agglomerative':
                linkage = extra_params.get('linkage', 'ward')
                model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
                labels = model.fit_predict(X_data)
            else:
                continue

            h, c, r, v = evaluate_clustering(y_true, labels)
            if v > best_score:
                best_score = v
                best_k = k
                best_labels = labels
        except Exception:
            continue
    return best_k, best_labels, best_score


In [88]:
# Сет настроек KernelPCA (sigmoid)
kpca_n_components_range = range(2, min(5, X_scaled.shape[1]) + 1)
kpca_gamma_range = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
kpca_coef0_range = [-1.0, 0.0, 0.5, 1.0, 2.0]

# Кластерные параметры
n_clusters_range = range(2, 7)

all_results = []

# Основной перебор: для каждого набора гиперпараметров KPCA выполняем GaussianMixture
for n_comp, gamma, coef0 in product(kpca_n_components_range, kpca_gamma_range, kpca_coef0_range):
    kpca_params_str = f"n_components={n_comp}, kernel=sigmoid, gamma={gamma}, coef0={coef0}"
    try:
        kpca = KernelPCA(n_components=n_comp, kernel='sigmoid', gamma=gamma, coef0=coef0, fit_inverse_transform=False)
        X_kpca = kpca.fit_transform(X_scaled)
    except Exception as e:
        print(f"KPCA error ({kpca_params_str}): {e}. Skipping.")
        continue

    # DataFrame
    comp_names = [f'Comp_{i+1}' for i in range(X_kpca.shape[1])]
    X_kpca_df = pd.DataFrame(X_kpca, columns=comp_names, index=X_scaled.index)

    # GaussianMixture: перебираем n_components явно
    for k in n_clusters_range:
        try:
            gmm = GaussianMixture(n_components=k, random_state=123, n_init=10)
            gmm.fit(X_kpca)
            labels = gmm.predict(X_kpca)
            h, c, r, v = evaluate_clustering(y, labels)
            all_results.append({
                'DR_Method': 'KernelPCA',
                'DR_Hyperparameter': kpca_params_str,
                'DR_n_components': n_comp,
                'Clustering_Algo': 'GaussianMixture',
                'Clustering_Hyperparameter': f"n_components={k}",
                'Optimal_k_found': k,
                'Homogeneity': h,
                'Completeness': c,
                'Rand Score': r,
                'V_measure': v
            })
        except Exception as e:
            print(f"GMM error ({kpca_params_str}, n_components={k}): {e}. Skipping.")
            continue

In [89]:
# Финальная сводка
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by='V_measure', ascending=False, ignore_index=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
print("\nTop-10 найденных конфигураций (по V_measure):")
print(results_df.head(10).to_string(index=False))




Top-10 найденных конфигураций (по V_measure):
DR_Method                                      DR_Hyperparameter  DR_n_components Clustering_Algo Clustering_Hyperparameter  Optimal_k_found  Homogeneity  Completeness  Rand Score  V_measure
KernelPCA n_components=5, kernel=sigmoid, gamma=0.01, coef0=-1.0                5 GaussianMixture            n_components=5                5     0.701844      0.689779    0.891252   0.695759
KernelPCA  n_components=5, kernel=sigmoid, gamma=0.01, coef0=0.0                5 GaussianMixture            n_components=5                5     0.684164      0.659146    0.886117   0.671422
KernelPCA  n_components=5, kernel=sigmoid, gamma=0.05, coef0=0.0                5 GaussianMixture            n_components=5                5     0.634990      0.654597    0.863063   0.644644
KernelPCA n_components=4, kernel=sigmoid, gamma=0.01, coef0=-1.0                4 GaussianMixture            n_components=5                5     0.653665      0.633680    0.874756   0.64351

In [90]:
# Выберем лучший результат и визуализируем
if not results_df.empty:
    best = results_df.loc[0]
    print("\nЛучший результат:")
    print(best.to_string())
    # Формируем kpca из строки в results_df
    best_dr_params = best['DR_Hyperparameter']
    def parse_kpca_params(param_str):
        params = {}
        for part in param_str.split(','):
            if '=' in part:
                k, v = part.split('=')
                k = k.strip()
                v = v.strip()
                try:
                    if '.' in v:
                        params[k] = float(v)
                    else:
                        params[k] = int(v)
                except ValueError:
                    params[k] = v
        return params

    kp = parse_kpca_params(best_dr_params)
    n_comp_best = int(kp.get('n_components', 2))
    gamma_best = float(kp.get('gamma', 0.1))
    coef0_best = float(kp.get('coef0', 0.0))

    kpca_best = KernelPCA(n_components=n_comp_best, kernel='sigmoid', gamma=gamma_best, coef0=coef0_best)
    X_best = kpca_best.fit_transform(X_scaled)
    comp_names = [f'Comp_{i+1}' for i in range(X_best.shape[1])]
    X_best_df = pd.DataFrame(X_best, columns=comp_names, index=X_scaled.index)

    # Воссоздадим лучший алгоритм и метки
    algo = best['Clustering_Algo']
    params_s = best['Clustering_Hyperparameter']
    best_labels = None
    try:
        if algo == 'GaussianMixture':
            k = int(params_s.split('=')[1])
            gm = GaussianMixture(n_components=k, random_state=123, n_init=10)
            gm.fit(X_best)
            best_labels = gm.predict(X_best)
        elif algo == 'KMeans':
            k = int(params_s.split('=')[1])
            km = KMeans(n_clusters=k, n_init=10, random_state=123)
            best_labels = km.fit_predict(X_best)
        elif algo == 'AgglomerativeClustering':
            k = int(params_s.split('n_clusters=')[1].split(',')[0])
            linkage = params_s.split('linkage=')[1]
            ag = AgglomerativeClustering(n_clusters=k, linkage=linkage)
            best_labels = ag.fit_predict(X_best)
        elif algo == 'DBSCAN':
            eps = float(params_s.split('eps=')[1].split(',')[0])
            min_s = int(params_s.split('min_samples=')[1])
            db = DBSCAN(eps=eps, min_samples=min_s)
            best_labels = db.fit_predict(X_best)
    except Exception as e:
        print("Ошибка при воссоздании лучшего алгоритма для визуализации:", e)



Лучший результат:
DR_Method                                                            KernelPCA
DR_Hyperparameter            n_components=5, kernel=sigmoid, gamma=0.01, co...
DR_n_components                                                              5
Clustering_Algo                                                GaussianMixture
Clustering_Hyperparameter                                       n_components=5
Optimal_k_found                                                              5
Homogeneity                                                           0.701844
Completeness                                                          0.689779
Rand Score                                                            0.891252
V_measure                                                             0.695759


In [91]:

plot_3d(X_best_df, y, best_labels, algo, params_s + "  ||  KPCA: " + best_dr_params, feature_names=comp_names[:3])


## Итог:
Лучший результат:
KernelPCA (n_components=5, kernel=sigmoid, gamma=0.01, coef0=-1.0) с алгоритмом
GaussianMixture. Оптимальное значение кластеров найдено 5.

Метрики
* Homogeneity:                                                           0.701844
* Completeness:                                                          0.689779
* Rand Score:                                                            0.891252
* V_measure:                                                            0.695759